In [ ]:
## Dependencies

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
warnings.filterwarnings('ignore')

## Model Definition

### Defining function for Model used, Train Model, and Cross Validations

In [ ]:
# ============================================================================
# STEP 1: DEFINE ALL MODELS
# ============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from xgboost import XGBClassifier
from sklearn.svm import SVC


def get_all_models():
    """
    Define all models to be tested
    Returns dictionary of models grouped by type
    """
    models = {
        # =====================================================================
        # SUPERVISED MODELS (IMBALANCED DATA)
        # =====================================================================
        "supervised_imbalanced": {
            "Random Forest (Weighted)": RandomForestClassifier(
                n_estimators=300,  # more trees → more stability for noisy features
                max_depth=4,  # allow trees to grow, later tune if needed
                min_samples_split=2,  # standard
                min_samples_leaf=2,  # standard; can increase to reduce overfitting
                max_features="sqrt",  # recommended for classification
                bootstrap=True,  # standard bootstrap RF
                class_weight="balanced",  # handles imbalance using y distribution
                n_jobs=-1,
                random_state=42,
            ),
            "XGBoost (Weighted)": XGBClassifier(
                n_estimators=200,
                learning_rate=0.1,
                max_depth=4,  # recommended
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=246 / 18,  # imbalance correction
                eval_metric="logloss",
                random_state=42,
            ),
            "Logistic Regression (Weighted)": LogisticRegression(
                class_weight="balanced",  # handles imbalance automatically
                penalty="l2",  # standard, stable regularization
                C=1.0,  # baseline inverse regularization strength
                solver="liblinear",  # safe for small datasets
                max_iter=2000,  # high enough for convergence
                random_state=42,
            ),
            "Decision Tree (Weighted)": DecisionTreeClassifier(
                criterion="gini",  # default and stable
                max_depth=4,  # shallow → avoids overfitting
                min_samples_leaf=2,  # stabilizes splits
                class_weight="balanced",  # compensate imbalance
                random_state=42,
            ),
            "KNN (Imbalanced)": KNeighborsClassifier(
                n_neighbors=5,  # standard, stable
                weights="distance",  # better for imbalanced data
                metric="minkowski",
                p=1,  # Euclidean distance
            ),
            "SVM_linear": SVC(
                kernel='linear', 
                class_weight='balanced', 
                probability=True, 
                random_state=42),
            "SVM_rbf": SVC(
                kernel='rbf', 
                class_weight='balanced', 
                probability=True, 
                random_state=42)
        },
        
    }

    return models


# ============================================================================
# STEP 2: TRAIN ALL MODELS
# ============================================================================


def train_all_models(X_train_scaled, y_train):
    """
    Train all models and return trained models with metadata
    """
    models = get_all_models()
    trained_models = {}

    print("\n" + "=" * 60)
    print("TRAINING ALL MODELS")
    print("=" * 60)

    # Train supervised models on imbalanced data
    print("\nTraining Supervised Models (Imbalanced Data)...")
    for name, model in models["supervised_imbalanced"].items():
        print(f"  - Training {name}...")
        model.fit(X_train_scaled, y_train)
        trained_models[name] = {"model": model, "type": "supervised", "trained": True}

    print("\nAll models trained successfully!")
    return trained_models

### CROSS VALIDATION METHOD


In [ ]:
# ============================================================================
# STEP 3: METRICS TABLE WITH OOF THRESHOLD TUNING
# ============================================================================

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, precision_recall_curve
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.base import clone
import pandas as pd
import numpy as np

def cv_metrics_table(trained_models, X, y):
    """
    Build metrics table with OOF threshold tuning (per model):
    - Supervised models         → 5-fold CV (StratifiedKFold)
    - Threshold per model       → chosen to maximize F1 on OOF predictions
    - Reported metrics          → based on tuned threshold
    """
    X = np.asarray(X)
    y = np.asarray(y)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rows = []

    for name, entry in trained_models.items():
        base_model = entry['model']
        mtype      = entry['type']

        # ---------------------------------------------------------
        # 1. SUPERVISED MODELS → cross-validation with OOF proba
        # ---------------------------------------------------------
        if mtype in ["supervised", "supervised_smote"]:
            if mtype == "supervised":
                estimator = clone(base_model)

            elif mtype == "supervised_smote":
                # if the model is already a pipeline with SMOTE inside,
                # do NOT add another SMOTE outside
                if isinstance(base_model, ImbPipeline):
                    estimator = clone(base_model)
                else:
                    estimator = ImbPipeline([
                        ('smote', SMOTE(random_state=42)),
                        ('clf', clone(base_model))
                    ])

            # Out-of-fold predicted probabilities for the positive class
            y_pred_proba = cross_val_predict(
                estimator, X, y, cv=cv, method='predict_proba'
            )[:, 1]

        else:
            # skip non-supervised entries
            continue

        # ---------------------------------------------------------
        # 2. THRESHOLD TUNING (on OOF probabilities)
        # ---------------------------------------------------------
        precision_arr, recall_arr, thresholds = precision_recall_curve(y, y_pred_proba)

        # F1 for each threshold (ignore potential division by zero)
        # f1_scores = 2 * precision_arr * recall_arr / (precision_arr + recall_arr + 1e-8)

        # Best threshold index (note: thresholds has len = len(precision)-1)
        # best_idx = np.argmax(f1_scores[:-1])  # ignore last point where threshold is undefined
        # best_threshold = thresholds[best_idx]
        best_threshold = 0.5 # default fallback

        # Use tuned threshold to convert probs to labels
        y_pred = (y_pred_proba >= best_threshold).astype(int)

        # ---------------------------------------------------------
        # 3. METRICS WITH TUNED THRESHOLD
        # ---------------------------------------------------------
        tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

        precision = precision_score(y, y_pred, zero_division=0)
        recall    = recall_score(y, y_pred, zero_division=0)
        f1        = f1_score(y, y_pred, zero_division=0)

        # ROC-AUC is threshold-free → still computed on probabilities
        rocauc = roc_auc_score(y, y_pred_proba)

        rows.append({
            "Model": name,
            "Type": mtype,
            "BestThreshold": best_threshold,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "ROC-AUC": rocauc,
            "True Positives": tp,
            "False Positives": fp,
            "False Negatives": fn,
            "True Negatives": tn,
        })

    df = pd.DataFrame(rows).sort_values(by="F1-Score", ascending=False)
    return df


### Test Metrics Table

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

def test_metrics_table(trained_models, cv_summary, X_test, y_test):
    """
    Evaluate tuned models on a TEST set using the thresholds
    previously selected via CV on the TRAIN set.

    Parameters
    ----------
    trained_models : dict
        Same structure you used before: {name: {'model': ..., 'type': ...}}.
        These models are already fitted on the TRAIN data.
    cv_summary : pd.DataFrame
        Output of cv_metrics_table on the TRAIN set, containing
        at least columns ['Model', 'BestThreshold'].
    X_test, y_test : array-like
        Held-out test data (no CV here).

    Returns
    -------
    pd.DataFrame
        Metrics on the test set for each model, using its CV-tuned threshold.
    """
    X_test = np.asarray(X_test)
    y_test = np.asarray(y_test)

    rows = []

    for name, entry in trained_models.items():
        base_model = entry['model']
        mtype      = entry['type']

        if mtype not in ["supervised", "supervised_smote"]:
            # skip anomaly / unsupervised etc.
            continue

        # 1) Get the best threshold from TRAIN CV summary
        row = cv_summary.loc[cv_summary["Model"] == name]
        if row.empty:
            # model was not in cv_summary_1 (or got filtered out)
            continue
        best_threshold = row["BestThreshold"].values[0]

        # 2) Predict probabilities on the TEST set
        #    (for SMOTE pipelines, you should have refit the pipeline on TRAIN
        #     before building `trained_models`).
        y_proba = base_model.predict_proba(X_test)[:, 1]

        # 3) Apply threshold
        y_pred = (y_proba >= best_threshold).astype(int)

        # 4) Compute metrics
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        precision = precision_score(y_test, y_pred, zero_division=0)
        recall    = recall_score(y_test, y_pred, zero_division=0)
        f1        = f1_score(y_test, y_pred, zero_division=0)
        rocauc    = roc_auc_score(y_test, y_proba)

        rows.append({
            "Model": name,
            "Type": mtype,
            "BestThreshold": best_threshold,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "ROC-AUC": rocauc,
            "True Positives": tp,
            "False Positives": fp,
            "False Negatives": fn,
            "True Negatives": tn,
        })

    return pd.DataFrame(rows).sort_values(by="F1-Score", ascending=False)


### Model Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


def tune_random_forest_weighted(X_train_scaled, y_train):
    """
    Hyperparameter tuning for Random Forest (Weighted).
    Returns best estimator, best_params, best_score.
    """
    rf_base = RandomForestClassifier(
        class_weight="balanced",
        max_depth=X_train_scaled.shape[1],
        random_state=42,
        n_jobs=-1,
    )

    param_grid = {
        "n_estimators": [100, 200, 500],
        # 'max_depth':         [2, 3, 4, None],
        "min_samples_split": [2, 3, 5],
        "min_samples_leaf": [2, 3, 5],
        "max_features": ["sqrt", "log2", 0.5],
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    grid = GridSearchCV(
        estimator=rf_base,
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )

    print("\n===== Tuning Random Forest (Weighted) =====")
    grid.fit(X_train_scaled, y_train)

    print("Best RF params:", grid.best_params_)
    print("Best RF CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


def tune_xgboost_weighted(X_train_scaled, y_train):
    """
    Hyperparameter tuning for XGBoost (Weighted).
    Assumes binary labels {0: healthy, 1: leakage}.
    Returns best estimator, best_params, best_score.
    """
    # Base model (scale_pos_weight will be tuned)
    xgb_base = XGBClassifier(
        eval_metric="logloss",
        random_state=42,
        n_estimators=200,
        max_depth=X_train_scaled.shape[1],
        # early_stopping_rounds=20,
        use_label_encoder=False,  # optional depending on xgboost version
    )

    # compute approximate neg/pos ratio:
    n_pos = (y_train == 1).sum()
    n_neg = (y_train == 0).sum()
    ratio = n_neg / max(n_pos, 1)

    param_grid = {
        "n_estimators": [100, 200],
        "learning_rate": [0.01, 0.05, 0.1],
        # 'max_depth':         [2, 3, 4],
        "min_child_weight": [3, 4, 5],
        "subsample": [0.6, 0.8],
        "colsample_bytree": [0.6, 0.8],
        "gamma": [0.0, 0.1, 0.5],
        # tune around the empirical imbalance ratio
        "scale_pos_weight": [0.5 * ratio, ratio],
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    grid = GridSearchCV(
        estimator=xgb_base,
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )

    print("\n===== Tuning XGBoost (Weighted) =====")
    grid.fit(X_train_scaled, y_train)

    print("Best XGB params:", grid.best_params_)
    print("Best XGB CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


##### Logistic Regression Tuning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold


def tune_logistic_regression_weighted(X_train_scaled, y_train):
    """
    Hyperparameter tuning for Logistic Regression (Weighted).
    Uses class_weight='balanced' to handle class imbalance.
    Returns best estimator, best_params, best_score.
    """

    lr_base = LogisticRegression(
        class_weight="balanced",
        solver="liblinear",  # robust for small datasets
        max_iter=500,
        random_state=42,
    )

    param_grid = {
        "C": [0.001, 0.01, 0.1, 1, 10],
        "penalty": ["l1", "l2"],
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    grid = GridSearchCV(
        estimator=lr_base,
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )

    print("\n===== Tuning Logistic Regression (Weighted) =====")
    grid.fit(X_train_scaled, y_train)

    print("Best LR params:", grid.best_params_)
    print("Best LR CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


######## KNN Tuning
from sklearn.neighbors import KNeighborsClassifier


def tune_knn_classifier(X_train_scaled, y_train):
    """
    Hyperparameter tuning for K-Nearest Neighbors classifier.
    Returns best estimator, best_params, best_score.
    """

    knn_base = KNeighborsClassifier()

    param_grid = {
        "n_neighbors": [5, 6, 7, 10, 15],
        "weights":["distance"],
        "metric": ["euclidean", "manhattan", "minkowski"],
        "p": [1, 2],  # p=1 (Manhattan), p=2 (Euclidean)
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    grid = GridSearchCV(
        estimator=knn_base,
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )

    print("\n===== Tuning KNN Classifier =====")
    grid.fit(X_train_scaled, y_train)

    print("Best KNN params:", grid.best_params_)
    print("Best KNN CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


########### Decision Tree tuning
from sklearn.tree import DecisionTreeClassifier


def tune_decision_tree_weighted(X_train_scaled, y_train):
    """
    Hyperparameter tuning for Decision Tree (Weighted).
    Returns best estimator, best_params, best_score.
    """

    dt_base = DecisionTreeClassifier(
        class_weight="balanced", 
        max_depth=X_train_scaled.shape[1], 
        random_state=42
    )

    param_grid = {
        "criterion": ["gini", "entropy", "log_loss"],
        # 'max_depth': [1, 2, 3, 4, 5],
        "min_samples_split": [2, 3, 4, 5, 10],
        "min_samples_leaf": [1, 2, 3, 4],
        "max_features": ["sqrt", "log2", None],
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    grid = GridSearchCV(
        estimator=dt_base,
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )

    print("\n===== Tuning Decision Tree (Weighted) =====")
    grid.fit(X_train_scaled, y_train)

    print("Best DT params:", grid.best_params_)
    print("Best DT CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


def tune_logistic_regression_smote(X_train, y_train):
    """
    Hyperparameter tuning for Logistic Regression with SMOTE.
    SMOTE is applied *inside* CV via an imblearn Pipeline.
    Returns best_estimator_, best_params_, best_score_.
    """

    # --- Base components ---
    smote = SMOTE(
        sampling_strategy="auto",  # can be tuned
        k_neighbors=5,  # can be tuned
        random_state=42,
    )

    lr_base = LogisticRegression(
        solver="liblinear",  # supports l1 and l2
        max_iter=500,
        random_state=42,
        # NOTE: do NOT use class_weight='balanced' here,
        # SMOTE already rebalances the data.
    )

    pipe = ImbPipeline(steps=[("smote", smote), ("lr", lr_base)])

    # --- Hyperparameter grid ---
    param_grid = {
        # Logistic Regression hyperparameters
        "lr__C": [0.01, 0.1, 1, 10, 100],
        "lr__penalty": ["l1", "l2"],
        # Optional: also tune SMOTE itself
        # (you can comment these out if you want to keep SMOTE fixed)
        "smote__k_neighbors": [3, 5, 7],
        "smote__sampling_strategy": ["auto", 0.75, 1.0],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="f1",  # same metric as your weighted version
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )

    print("\n===== Tuning Logistic Regression (SMOTE) =====")
    grid.fit(X_train, y_train)

    print("Best LR+SMOTE params:", grid.best_params_)
    print("Best LR+SMOTE CV F1:", grid.best_score_)

    # This object (pipeline) already includes the *tuned* SMOTE + LR
    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_

# Start of Main Algorithm

## Data preparation

In [ ]:
# ======================================================================
# STEP 1: LOAD AND PREPARE DATA (REFERENCE + MULTI-KIT MONORAIL)
# ======================================================================

def load_data(model_path, monorail_paths):
    """
    Load and prepare reference (model) data and Monorail data (one or more kits),
    align common columns, and return a single combined DataFrame.

    Parameters
    ----------
    model_path : str
        Path to model.csv (reference experimental campaign).
    monorail_paths : str or list of str
        Path or list of paths to Monorail TestBrakefinal_data_kitXX.csv files.

    Returns
    -------
    df_base : pandas.DataFrame
        Combined DataFrame with:
        - aligned common columns between reference and Monorail,
        - binary label (0/1) where available,
        - 'Source' column (kit ID or 0 for reference),
        - 'DataSource' column (0 = reference, 1 = Monorail).
    """

    # --------------------------------------------------------------
    # Helper: load and clean ONE Monorail file
    # --------------------------------------------------------------
    def load_Monorail(filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)

        # Keep only standard braking
        if 'Non_Standard_Braking' in df.columns:
            df = df[df['Non_Standard_Braking'] == 0]

        # Extract numeric kit ID from filename, e.g. "TestBrakefinal_data_kit06.csv" -> 6
        match = re.search(r'kit(\d+)', os.path.basename(filepath))
        source = int(match.group(1)) if match else -1
        df['Source'] = source

        # Convert "xx sec" string columns to float seconds where possible
        for col in df.select_dtypes(include='object'):
            try:
                df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
            except (AttributeError, ValueError):
                # AttributeError if column is not string-like; ValueError if some values cannot be cast
                continue

        return df

    # --------------------------------------------------------------
    # 1) REFERENCE DATA: load, label, aggregate
    # --------------------------------------------------------------
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

    # Binary label from malfunction code
    leakage_codes = ['C', 'D', 'E', 'F', 'G']
    df_reference['LeakageLabel'] = np.where(
        df_reference['Malfunction'].isin(leakage_codes),
        'Combined leakage',
        'Healthy'
    )

    # Add Source = 0 for reference campaign
    df_reference['Source'] = 0

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay':      ['Brake_timing_delay_exp',      'Release_timing_delay_exp'],
        'Total_energy_delay':      ['Brake_energy_delay_exp',      'Release_energy_delay_exp'],
        'Total_power_delay':       ['Brake_power_delay_exp',       'Release_power_delay_exp'],
        'Total_power_efficiency':  ['Brake_power_efficiency_exp',  'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp',   'Release_energy_efficiency_exp']
    }

    for new_col, (c1, c2) in delay_eff_map.items():
        # If any of these columns are missing in some version of model.csv, guard with .get
        if c1 in df_reference.columns and c2 in df_reference.columns:
            df_reference[new_col] = df_reference[c1] + df_reference[c2]

    # Drop original per-phase columns (only those that actually exist)
    cols_to_drop = [c for pair in delay_eff_map.values() for c in pair if c in df_reference.columns]
    df_reference.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # Rename to your canonical names
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp':  'Buildup_end_pressure_delay',
        'Weight':                          'WV_MeanPressure',
        'Brake_action':                    'EmergencyBrake_action'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # --------------------------------------------------------------
    # 2) MONORAIL DATA: load one or more kit files
    # --------------------------------------------------------------
    if isinstance(monorail_paths, str):
        monorail_paths = [monorail_paths]

    dfs_mono = [load_Monorail(fp) for fp in monorail_paths]
    df_data = pd.concat(dfs_mono, ignore_index=True)

    # --------------------------------------------------------------
    # 3) ALIGN STRUCTURES AND COMBINE
    # --------------------------------------------------------------
    # Ensure 'Source' is integer in both
    df_reference['Source'] = df_reference['Source'].astype(int)
    df_data['Source']      = df_data['Source'].astype(int)

    # Columns common to BOTH datasets
    common_cols = df_reference.columns.intersection(df_data.columns).tolist()

    # Subsets with only common columns + a DataSource flag
    df_reference_subset = df_reference[common_cols].copy()
    df_reference_subset['DataSource'] = 0  # 0 = reference campaign

    df_data_subset = df_data[common_cols].copy()
    df_data_subset['DataSource'] = 1       # 1 = Monorail (real-time) data

    # Stack reference + Monorail
    df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

    # Encode final label column (will be NaN for Monorail if it has no LeakageLabel)
    if 'LeakageLabel' in df_combined.columns:
        df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
        df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Combined leakage': 1})

    # Convert any remaining "xx sec" string columns to float (esp. from model.csv)
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except (AttributeError, ValueError):
            continue
    df_combined["WV_bin"] = df_combined["WV_MeanPressure"].apply(
    lambda p: np.nan if pd.isna(p) else (0 if p < 2 else (2 if p > 3 else 1))
    )
    df_base = df_combined.copy()
    return df_base

model_path = 'model.csv'
monorail_paths = [
    'TestBrakefinal_data_kit01.csv',
    'TestBrakefinal_data_kit06.csv',
    'TestBrakefinal_data_kit27.csv'
]

df = load_data(model_path, monorail_paths)

print(df.shape)
print(df['DataSource'].value_counts(dropna=False))
print(df['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail

In [ ]:
df.head()

In [ ]:
# ======================================================================
# STEP 2: PREPROCESS DATA FOR ML
# ======================================================================

from sklearn.model_selection import train_test_split

def preprocess_data(df, features, test_size):
    """
    Preprocess data: create WV_bin column, filter by bin==1, select features,
    split into train/test, and separate healthy samples for training.
    
    Parameters:
    - df: pandas DataFrame containing the data
    - features: list of feature names to use (e.g., ['Total_power_efficiency', 'FlowRate'])
    - test_size: proportion of the dataset to include in the test split
    
    Returns:
    - X_train, X_test, y_train, y_test, X_train_healthy
    """
    
    # Create WV_bin column
    df = df.copy()
    # Filter by WV_bin == 1 (pressure between 2 and 3)
    df_filt = df[df["WV_bin"] == 1].copy()
    
    # Validate feature selection
    missing = [f for f in features if f not in df_filt.columns]
    if missing:
        raise ValueError(f"The following features are not in the dataframe: {missing}")
    
    # Select features and labels
    X = df_filt[features]
    y = df_filt['label']
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )
    
    # Extract healthy training samples
    X_train_healthy = X_train[y_train == 0]
    
    print(f"Total samples: {len(df)}")
    print(f"Filtered samples (WV_bin==1): {len(df_filt)}")
    print(f"Training samples: {len(X_train)} (Healthy: {sum(y_train==0)}, Leakage: {sum(y_train==1)})")
    print(f"Training samples (healthy only): {len(X_train_healthy)}")
    print(f"Test samples: {len(X_test)} (Healthy: {sum(y_test==0)}, Leakage: {sum(y_test==1)})")
    
    return X_train, X_test, y_train, y_test, X_train_healthy

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

def test_metrics_from_final(final_models, final_thresholds, X_test, y_test):
    """
    Evaluate already-fitted final models on TEST set, using
    per-model thresholds that were tuned on the TRAIN set.

    Parameters
    ----------
    final_models : dict
        {model_name: fitted_estimator}
        (e.g. final_models_S1, final_models_S2, ...)
    final_thresholds : dict
        {model_name: best_threshold_from_train}
        (e.g. final_thresholds_S1, ...)
    X_test, y_test : array-like
        Held-out test set.

    Returns
    -------
    pd.DataFrame
        Test metrics for each model.
    """
    X_test = np.asarray(X_test)
    y_test = np.asarray(y_test)

    rows = []

    for name, estimator in final_models.items():
        if name not in final_thresholds:
            continue

        thr = final_thresholds[name]

        # probabilities on TEST
        proba = estimator.predict_proba(X_test)[:, 1]
        y_pred = (proba >= thr).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)
        roc  = roc_auc_score(y_test, proba)

        rows.append({
            "Model": name,
            "BestThreshold_train": thr,
            "Precision_Test": prec,
            "Recall_Test": rec,
            "F1_Test": f1,
            "ROC-AUC_Test": roc,
            "TP_Test": tp,
            "FP_Test": fp,
            "FN_Test": fn,
            "TN_Test": tn,
        })

    return pd.DataFrame(rows).sort_values(
        by="F1_Test", ascending=False
    ).reset_index(drop=True)


## Data Exploration Continues

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df['y_jitter'] = rng.normal(0, 0.02, size=len(df))

# ------------------------------------------------------------
# 2. Palettes
# ------------------------------------------------------------
palette_source = sns.color_palette("colorblind", n_colors=len(df['Source'].unique()))
label_palette = {
    0: (0.2, 0.4, 0.9, 0.35),   # RGBA → semi-transparent blue
    1: (1.0, 0.55, 0.0, 1.0),   # solid orange
}

# df_filtered = df[(df['EmergencyBrake_action'] == 1) & (df['WV_bin'] == 1)].copy()
df_filtered = df[(df['DataSource'] == 0) & (df['WV_bin'] == 1)].copy()
# df_filtered = df[(df['WV_bin'] == 1)].copy()
# df_filtered = df.copy()
# palette_label  = sns.color_palette("Set1", n_colors=len(df['label'].unique()))

# Map categories to colors
source_codes = df_filtered['Source'].astype('category').cat.codes
label_codes  = df_filtered['label'].astype('category').cat.codes

colors_source = [palette_source[c] for c in source_codes]
colors_label  = [label_palette[c]  for c in label_codes]

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_filtered['Total_power_efficiency'],
    df_filtered['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("Total Power Efficiency distribution by Kit Source")
axes[0].set_xlabel("Total Power Efficiency")
axes[0].set_yticks([])

# Source legend
sources = df_filtered['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")
axes[0].set_xlim(0, 8)
# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_filtered['Total_power_efficiency'],
    df_filtered['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("Total Power Efficiency distribution by Label")
axes[1].set_xlabel("Total Power Efficiency")
axes[1].set_yticks([])

# Label legend
labels = df_filtered['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Leakage Label", loc="upper right")
axes[1].set_xlim(0, 8)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df['y_jitter'] = rng.normal(0, 0.02, size=len(df))
# df_filtered = df[(df['Max_pressure_pipe'] < 0.8) & (df['WV_bin'] == 1)].copy()
df_filtered = df[(df['EmergencyBrake_action'] == 1) & (df['WV_bin'] == 1)].copy()
# df_filtered = df[(df['DataSource'] == 0) & (df['WV_bin'].isin([1]))].copy()
# df_filtered = df[(df['DataSource'] == 0) & (df['WV_bin'] == 1)].copy()
# df_filtered = df[(df['WV_bin'] == 1)].copy()
# ------------------------------------------------------------
# 2. Palettes
# ------------------------------------------------------------
palette_source = sns.color_palette("colorblind", n_colors=len(df['Source'].unique()))
label_palette = {
    0: (0.2, 0.4, 0.9, 0.35),   # RGBA → semi-transparent blue
    1: (1.0, 0.55, 0.0, 1.0),   # solid orange
}


# palette_label  = sns.color_palette("Set1", n_colors=len(df['label'].unique()))

# Map categories to colors
source_codes = df_filtered['Source'].astype('category').cat.codes
label_codes  = df_filtered['label'].astype('category').cat.codes

colors_source = [palette_source[c] for c in source_codes]
colors_label  = [label_palette[c]  for c in label_codes]

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_filtered['Buildup_end_pressure_delay'],
    df_filtered['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("Total Power Delay distribution by Kit Source")
axes[0].set_xlabel("Total Power Delay")
axes[0].set_yticks([])

# Source legend
sources = df_filtered['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")

# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_filtered['Buildup_end_pressure_delay'],
    df_filtered['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("Total Power Delay distribution by Label")
axes[1].set_xlabel("Total Power Delay")
axes[1].set_yticks([])

# Label legend
labels = df_filtered['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Leakage Label", loc="upper right")
axes[1].set_xlim(-1, 2)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

def plot_efficiency_vs_features(df, features, base_width=7, base_height=5, x_limits=None):
    """
    Plot Total_power_efficiency against each selected feature.
    Creates N separate figures (one per feature).
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data containing 'Total_power_efficiency', 'label', and selected features.
    features : list of str
        List of feature column names to plot against Total_power_efficiency.
    base_width : int, optional
        Width of each figure (default=7).
    base_height : int, optional
        Height of each figure (default=5).
    x_limits : tuple (min, max), optional
        Limits for the X-axis (Total_power_efficiency).
    """
    # Masks for labels
    mask_0 = df['label'] == 0
    mask_1 = df['label'] == 1
    
    for feature in features:
        plt.figure(figsize=(base_width, base_height))
        
        plt.scatter(
            df.loc[mask_0, 'Total_power_efficiency'],
            df.loc[mask_0, feature],
            color='blue', alpha=0.4, label='0'
        )
        plt.scatter(
            df.loc[mask_1, 'Total_power_efficiency'],
            df.loc[mask_1, feature],
            color='red', alpha=0.7, label='1'
        )
        
        plt.xlabel("Total_power_efficiency")
        plt.ylabel(feature)
        plt.title(f"Total_power_efficiency vs {feature}")
        plt.legend(title='label', bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # Apply X-axis limits if provided
        if x_limits is not None:
            plt.xlim(x_limits)
        
        plt.tight_layout()
        plt.show()

# Example usage:
df_filtered = df[df['WV_bin'] == 1]

selected_features = ["Total_power_delay", "WV_MeanPressure","Total_energy_efficiency","Std_delay_exp"]

# Limit X-axis between 0 and 100
plot_efficiency_vs_features(df_filtered, selected_features, x_limits=(0, 8))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df_filt = df[df["WV_bin"] == 1].copy()
df_filt['y_jitter'] = rng.normal(0, 0.02, size=len(df_filt))

# ------------------------------------------------------------
# 2. Palettes
# ------------------------------------------------------------
palette_source = sns.color_palette("colorblind", n_colors=len(df_filt['Source'].unique()))
label_palette = {
    0: (0.2, 0.4, 0.9, 0.35),   # RGBA → semi-transparent blue
    1: (1.0, 0.55, 0.0, 1.0),   # solid orange
}


# palette_label  = sns.color_palette("Set1", n_colors=len(df['label'].unique()))

# Map categories to colors
source_codes = df_filt['Source'].astype('category').cat.codes
label_codes  = df_filt['label'].astype('category').cat.codes

colors_source = [palette_source[c] for c in source_codes]
colors_label  = [label_palette[c]  for c in label_codes]

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_filt['Buildup_end_pressure_delay'],
    df_filt['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("Total Power Delay distribution by Kit Source")
axes[0].set_xlabel("Total Power Delay")
axes[0].set_yticks([])

# Source legend
sources = df_filt['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")

# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_filt['Buildup_end_pressure_delay'],
    df_filt['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("Total Power Delay distribution by Label")
axes[1].set_xlabel("Total Power Delay")
axes[1].set_yticks([])

# Label legend
labels = df_filt['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Leakage Label", loc="upper right")

plt.tight_layout()
plt.show()


## Algorithm 1 - Use only 2 Features

In [ ]:
# ============================================================================
# STEP 1: DATA PREPROCESSING TRAIN TEST SPLIT AND FEATURE SELECTION 
# ============================================================================
selected_features = ['Total_power_efficiency','Std_delay_exp'] 

[X_train_1, X_test_1, y_train, y_test, X_train_healthy_1] = preprocess_data(df, selected_features, test_size=0.2)
X_train_1.head()

In [ ]:
# ============================================================================
# STEP 2: IMPUTE + FEATURE SCALING
# ============================================================================

from sklearn.impute import SimpleImputer

def scale_features(X_train, X_test, X_train_healthy):
    """
    Impute missing values by median, then standardize features.
    Returns imputed+scaled arrays, plus fitted scaler and imputer.
    """
    # 1) Median imputation (fit only on training set)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)
    X_train_healthy_imp = imputer.transform(X_train_healthy)

    # 2) Standardization (fit only on imputed training set)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    X_train_healthy_scaled = scaler.transform(X_train_healthy_imp)

    return X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer

[X_train_scaled_1, X_test_scaled_1, X_train_healthy_scaled_1, scaler, imputer] = scale_features(X_train_1, X_test_1, X_train_healthy_1)

### Train the Model

In [ ]:
trained_models = train_all_models(X_train_scaled_1, y_train)

### Simple Model Training

#### Very Simple Model training with Cross Validation Proof
* KNN Model

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import precision_recall_curve

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold = 0
pred_proba = np.zeros(len(y_train))
for train_idx, val_idx in skf.split(X_train_1, y_train):
    X_tr = X_train_1.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]
    
    X_val = X_train_1.iloc[val_idx]
    y_val = y_train.iloc[val_idx]
    
    # Fit model on Train
    clf = KNeighborsClassifier(
                n_neighbors=5,       # will be tuned
                weights='distance',
                metric='minkowski',
                p=2)
    clf.fit(X_tr, y_tr)
    pred = clf.predict(X_val)
    pred_proba[val_idx] = clf.predict_proba(X_val)[:, 1]
    acc_score = accuracy_score(y_val, pred)
    auc_score = roc_auc_score(y_val, pred_proba[val_idx])
    f1_fold = f1_score(y_val, pred)
    print(f"========= Fold {fold} =========")
    print(
        f"Accuracy: {acc_score:.4f}, AUC: {auc_score:.4f}, F1: {f1_fold:.4f}"
    )
    fold += 1

# Determine best threshold based on F1 score
precision, recall, thresholds = precision_recall_curve(y_train, pred_proba)
f1 = 2 * precision * recall / (precision + recall + 1e-8)
best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]

# Out of fold Global evaluation
oof_pred = (pred_proba >= best_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_train, oof_pred).ravel()
f1score = f1_score(y_train, oof_pred)
rocauc = roc_auc_score(y_train, pred_proba)
print("\n========= Global Evaluation =========")
print(f"Best Threshold: {best_threshold:.4f}")
print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"F1 Score: {f1score:.4f}")
print(f"ROC AUC: {rocauc:.4f}")


In [ ]:
plt.figure(figsize=(7,6))
plt.plot(recall, precision, label="Precision–Recall Curve")
plt.scatter(recall[best_idx], precision[best_idx], color='red', s=80,
            label=f"Best Threshold = {best_threshold:.3f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(7,6))
plt.plot(thresholds, f1[:-1], label="F1 Score vs Threshold")  
plt.axvline(best_threshold, color='red', linestyle='--',
            label=f"Best Threshold = {best_threshold:.3f}")

plt.xlabel("Threshold")
plt.ylabel("F1 Score")
plt.title("F1 Score vs Threshold")
plt.legend()
plt.grid(True)
plt.show()


#### Very simple KNN Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance'],
    'p': [1, 2],  # 1=Manhattan, 2=Euclidean
    'metric': ['minkowski', 'euclidean', 'manhattan']
}

# Setup GridSearch with CV
knn = KNeighborsClassifier()
grid_search = GridSearchCV(
    knn, 
    param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',  # or 'roc_auc' depending on your priority
    n_jobs=-1,
    verbose=1
)

# Fit on training data
grid_search.fit(X_train_1, y_train)

# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV F1-Score:", grid_search.best_score_)

# Save best model
best_knn = grid_search.best_estimator_

In [ ]:
# Lets say KNN is the best model from previous steps
final_model = best_knn
pred_proba_oof = cross_val_predict(
    final_model,
    X_train_1,
    y_train,
    cv=5,
    method='predict_proba'
)[:,1]
precision, recall, thresholds = precision_recall_curve(y_train, pred_proba_oof)
f1_curve = 2 * precision * recall / (precision + recall + 1e-8)
best_idx = np.argmax(f1_curve)
best_threshold = thresholds[best_idx]
oof_pred = (pred_proba_oof >= best_threshold).astype(int)
f1_final = f1_score(y_train, oof_pred)

print(f"Best Threshold: {best_threshold:.4f}")
print(f"F1 Score: {f1_final:.4f}")

In [ ]:
# Predictions on test set
y_test_pred = final_model.predict(X_test_1)
y_test_pred_proba = final_model.predict_proba(X_test_1)[:, 1]

# Use tuned threshold instead of default 0.5
y_test_pred_tuned = (y_test_pred_proba >= best_threshold).astype(int)

print("\nClassification Report (tuned threshold):")
print(classification_report(y_test, y_test_pred_tuned))

print("\nConfusion Matrix (tuned threshold):")
print(confusion_matrix(y_test, y_test_pred_tuned))

test_f1 = f1_score(y_test, y_test_pred_tuned)
test_auc = roc_auc_score(y_test, y_test_pred_proba)  # AUC unchanged (threshold-free)

print(f"\nTest F1-Score (tuned): {test_f1:.4f}")
print(f"Test ROC-AUC: {test_auc:.4f}")


In [ ]:
cv_summary_1 = cv_metrics_table(
    trained_models, X_train_scaled_1, y_train
)

cv_summary_1

Try the UNTUNNED MODEL on Test Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# List of supervised models you want to visualize
best_models = [
    "KNN (Imbalanced)",
    "Logistic Regression (Weighted)",
    "Decision Tree (Weighted)",
    "Random Forest (Weighted)",
    "XGBoost (Weighted)",
]

# Convert test set to numpy
X = np.asarray(X_test_scaled_1)
y = np.asarray(y_test)

# --------------------------------------------------------
# LOOP THROUGH EACH MODEL AND CREATE A SEPARATE FIGURE
# --------------------------------------------------------
for model_name in best_models:

    # Retrieve model info
    model_info = trained_models[model_name]
    mtype = model_info['type']
    
    if mtype not in ('supervised', 'supervised_smote'):
        raise ValueError(f"{model_name} is not a supervised model.")

    base_model = model_info['model']

    # Retrieve best threshold from your CV summary table
    row = cv_summary_1.loc[
        cv_summary_1['Model'] == model_name
    ]
    if row.empty:
        raise ValueError(f"No CV summary row found for '{model_name}'.")

    best_threshold = row['BestThreshold'].iloc[0]

    # Predict on test set
    y_proba = base_model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= best_threshold).astype(int)

    # Confusion matrix
    cm = confusion_matrix(y, y_pred)
    max_count = cm.max()  # ensures consistent color scale

    # --------------------------------------------------------
    # PLOT — ONE FIGURE PER MODEL
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=(5, 4))

    im = ax.imshow(cm, cmap='coolwarm', vmin=0, vmax=max_count)

    ax.set_title(
        f"{model_name}\nThreshold = {best_threshold:.3f}",
        fontsize=12, pad=10
    )

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(['Pred Healthy', 'Pred Leakage'])
    ax.set_yticklabels(['True Healthy', 'True Leakage'])
    ax.grid(False)
    # Annotate values
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > max_count / 2 else 'black'
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color=color, fontsize=11)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Count")

    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import cross_val_predict

plt.figure(figsize=(7, 6))

models_to_plot = [
    "KNN (Imbalanced)",
    "Logistic Regression (Weighted)",
    "Decision Tree (Weighted)",
    "Random Forest (Weighted)",
    "XGBoost (Weighted)"
]

X = np.asarray(X_train_scaled_1)
y = np.asarray(y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name in models_to_plot:
    entry = trained_models[name]
    base_model = entry['model']
    mtype      = entry['type']

    # Build estimator
    if mtype == 'supervised':
        estimator = clone(base_model)
    elif mtype == 'supervised_smote':
        estimator = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', clone(base_model))
        ])
    else:
        print(f"Skipping anomaly model: {name}")
        continue

    # CV predicted probabilities
    y_scores = cross_val_predict(
        estimator, X, y,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    fpr, tpr, _ = roc_curve(y, y_scores)
    auc = roc_auc_score(y, y_scores)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", lw=2)

# Random baseline
plt.plot([0, 1], [0, 1], 'k--', label="Random classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Cross-validated ROC Curves for Multiple Models")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


### Metrics - F1 Score Table

### Hyperparameter Tuning of 1 Features

In [ ]:
best_lr, lr_params, lr_f1, lr_cv = tune_logistic_regression_weighted(X_train_scaled_1, y_train)
best_knn, knn_params, knn_f1, knn_cv = tune_knn_classifier(X_train_scaled_1, y_train)
best_dt, dt_params, dt_f1, dt_cv = tune_decision_tree_weighted(X_train_scaled_1, y_train)


In [ ]:
# Run XGboost tuning
best_xgb_est_tpe, best_xgb_params_tpe, best_xgb_score_tpe, xgb_cv = tune_xgboost_weighted(
    X_train_scaled_1, y_train
)

# Run Random Forest tuning
best_rf_est_tpe, best_rf_params_tpe, best_rf_score_tpe, rf_cv = tune_random_forest_weighted(
    X_train_scaled_1, y_train
)

In [ ]:
best_lr_smote, best_params_smote, best_f1_smote, lrsmote_cv = tune_logistic_regression_smote(
    X_train_scaled_1, y_train
)

In [ ]:
# Replace only the model with the best estimator
tuned_models = trained_models.copy()
tuned_models['Logistic Regression (Weighted)']['model'] = best_lr
tuned_models['KNN (Imbalanced)']['model'] = best_knn
tuned_models['Decision Tree (Weighted)']['model'] = best_dt
tuned_models['XGBoost (Weighted)']['model'] = best_xgb_est_tpe
tuned_models['Random Forest (Weighted)']['model'] = best_rf_est_tpe

# Evaluate
cv_results_1_feat_tuned = cv_metrics_table(
    tuned_models, X_train_scaled_1, y_train
)
cv_results_1_feat_tuned

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Clean model names if needed (important for merging)
df_before = cv_summary_1.copy()
df_after = cv_results_1_feat_tuned.copy()

df_before['Model'] = df_before['Model'].str.strip()
df_after['Model'] = df_after['Model'].str.strip()

# Merge by model name
df_compare = pd.merge(
    df_before,
    df_after,
    on='Model',
    suffixes=('_before', '_after')
)

metrics = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC']

for m in metrics:
    df_compare[f'{m}_delta'] = df_compare[f'{m}_after'] - df_compare[f'{m}_before']

cols_show = [
    'Model',
    'BestThreshold_before', 'BestThreshold_after',
    'Precision_before', 'Precision_after', 'Precision_delta',
    'Recall_before', 'Recall_after', 'Recall_delta',
    'F1-Score_before', 'F1-Score_after', 'F1-Score_delta',
    'ROC-AUC_before', 'ROC-AUC_after', 'ROC-AUC_delta',
]

comparison_table = df_compare[cols_show].sort_values(by='F1-Score_delta', ascending=False)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

metrics = ['Precision', 'Recall', 'F1-Score']

# x locations for the groups
models = df_compare['Model'].tolist()
x = np.arange(len(models))
width = 0.35  # bar width

for metric in metrics:
    before_vals = df_compare[f'{metric}_before']
    after_vals  = df_compare[f'{metric}_after']

    plt.figure(figsize=(10, 5))
    
    # Bars: before and after, side by side
    plt.bar(x - width/2, before_vals, width, label='Before tuning')
    plt.bar(x + width/2, after_vals,  width, label='After tuning')
    
    plt.xticks(x, models, rotation=45, ha='right')
    plt.ylabel(metric)
    plt.title(f'{metric} (Training CV) – Before vs After Tuning')
    plt.ylim(0, 1.05)  # assuming metrics in [0, 1]
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
def fit_models(models_dict, X_train, y_train):
    for name, entry in models_dict.items():
        print(f"Training {name}...")
        entry['model'].fit(X_train, y_train)
    return models_dict

# Usage
tuned_models = fit_models(tuned_models, X_train_scaled_1, y_train)

In [ ]:
# On test set (reuse best thresholds if any)
test_results_1 = test_metrics_table(tuned_models, cv_results_1_feat_tuned, X_test_scaled_1, y_test)
test_results_1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# List of supervised models you want to visualize
best_models = [
    "KNN (Imbalanced)",
    "Logistic Regression (Weighted)",
    "Decision Tree (Weighted)",
    "Random Forest (Weighted)",
    "XGBoost (Weighted)"
]

# Convert test set to numpy
X = np.asarray(X_test_scaled_1)
y = np.asarray(y_test)

# --------------------------------------------------------
# LOOP THROUGH EACH MODEL AND CREATE A SEPARATE FIGURE
# --------------------------------------------------------
for model_name in best_models:

    # Retrieve model info
    model_info = tuned_models[model_name]
    mtype = model_info['type']
    
    if mtype not in ('supervised', 'supervised_smote'):
        raise ValueError(f"{model_name} is not a supervised model.")

    base_model = model_info['model']

    # Retrieve best threshold from your CV summary table
    row = cv_results_1_feat_tuned.loc[
        cv_results_1_feat_tuned['Model'] == model_name
    ]
    if row.empty:
        raise ValueError(f"No CV summary row found for '{model_name}'.")

    best_threshold = row['BestThreshold'].iloc[0]

    # Predict on test set
    y_proba = base_model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= best_threshold).astype(int)

    # Confusion matrix
    cm = confusion_matrix(y, y_pred)
    max_count = cm.max()  # ensures consistent color scale

    # --------------------------------------------------------
    # PLOT — ONE FIGURE PER MODEL
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=(5, 4))

    im = ax.imshow(cm, cmap='coolwarm', vmin=0, vmax=max_count)

    ax.set_title(
        f"{model_name}\nThreshold = {best_threshold:.3f}",
        fontsize=12, pad=10
    )

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(['Pred Healthy', 'Pred Leakage'])
    ax.set_yticklabels(['True Healthy', 'True Leakage'])
    ax.grid(False)
    # Annotate values
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > max_count / 2 else 'black'
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color=color, fontsize=11)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Count")

    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# Get the trained decision tree model
tree = tuned_models['Decision Tree (Weighted)']['model']

# Plot the tree
plt.figure(figsize=(16, 10))  # Adjust size as needed
plot_tree(
    tree,
    feature_names=selected_features,
    class_names=['Healthy', 'Leakage'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree Visualization")
plt.show()

## Feature Sensitivity Analysis based on F1 Scores

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    f1_score,
    precision_recall_curve,
)
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Union
import matplotlib.pyplot as plt


def feature_sensitivity_analysis(
    X: Union[pd.DataFrame, np.ndarray],
    y: np.ndarray,
    feature_names: List[str],
    models: Dict[str, object],
    max_features: Optional[int] = None,
    cv_splits: int = 5,
    step: int = 1,
    random_state: int = 42,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Incremental feature sensitivity analysis with F1 + threshold tuning.

    For each model and each feature-count k:
      - uses first k features in `feature_names`
      - computes out-of-fold predicted probabilities (StratifiedKFold)
      - tunes the threshold to maximize F1 using precision-recall curve
      - reports the tuned F1 as 'Score'

    This matches the behaviour of `cv_metrics_table`, but per feature subset.
    """

    # Sanity & limits
    if max_features is None:
        max_features = len(feature_names)
    max_features = min(max_features, len(feature_names))

    # Ensure numpy arrays for y
    y = np.asarray(y)

    cv = StratifiedKFold(
        n_splits=cv_splits,
        shuffle=True,
        random_state=random_state
    )

    results = []

    for model_name, base_estimator in models.items():
        if verbose:
            print("\n" + "=" * 50)
            print(f"Analyzing: {model_name}")
            print("=" * 50)

        for k in range(1, max_features + 1, step):
            subset_features = feature_names[:k]
            X_subset = _subset_features(X, subset_features, feature_names)

            # ------------------------------------------------------------------
            # 1) Out-of-fold predicted probabilities (like in cv_metrics_table)
            # ------------------------------------------------------------------
            estimator = clone(base_estimator)

            y_proba_oof = cross_val_predict(
                estimator,
                X_subset,
                y,
                cv=cv,
                method="predict_proba"
            )

            # assume binary classification: take positive-class probability
            if y_proba_oof.ndim == 2:
                y_scores = y_proba_oof[:, 1]
            else:
                raise ValueError(
                    f"Estimator {model_name} did not return a proper prob. array."
                )

            # ------------------------------------------------------------------
            # 2) Threshold tuning by PR-curve (same logic as cv_metrics_table)
            # ------------------------------------------------------------------
            precision_arr, recall_arr, thresholds = precision_recall_curve(y, y_scores)
            f1_scores = 2 * precision_arr * recall_arr / (precision_arr + recall_arr + 1e-8)

            # thresholds has len = len(precision_arr) - 1
            best_idx = np.argmax(f1_scores[:-1])
            best_threshold = thresholds[best_idx]

            # Hard labels at tuned threshold
            y_pred = (y_scores >= best_threshold).astype(int)
            tuned_f1 = f1_score(y, y_pred, zero_division=0)

            results.append({
                "Model": model_name,
                "NumFeatures": k,
                "Score": tuned_f1,
                "BestThreshold": best_threshold,
                "Features": ", ".join(subset_features[:3]) +
                            (f" (+{k-3} more)" if k > 3 else "")
            })

            if verbose and (k % 5 == 0 or k == 1 or k == max_features):
                print(f"  {k:3d} features | F1 (tuned): {tuned_f1:.4f} "
                      f"| thr: {best_threshold:.3f}")

    results_df = pd.DataFrame(results)

    if verbose:
        print("\n" + "=" * 50)
        print("Analysis Complete!")
        print("=" * 50 + "\n")
        _print_summary(results_df)

    return results_df


def _subset_features(X, subset_features, all_feature_names):
    """Extract feature subset from X."""
    if isinstance(X, pd.DataFrame):
        return X[subset_features].values
    else:
        indices = [all_feature_names.index(f) for f in subset_features]
        return X[:, indices]


def _print_summary(df: pd.DataFrame):
    """Print summary of best results per model."""
    print("Best tuned F1 by Model:")
    print("-" * 70)

    for model in df["Model"].unique():
        model_df = df[df["Model"] == model]
        best_idx = model_df["Score"].idxmax()
        best = model_df.loc[best_idx]

        print(f"{model:25s} | {best['NumFeatures']:3d} features | "
              f"F1: {best['Score']:.4f} | thr: {best['BestThreshold']:.3f}")


def plot_sensitivity_analysis(
    results_df: pd.DataFrame,
    figsize: tuple = (12, 5),
    save_path: Optional[str] = None,
    metric_label: str = "F1-score",
):
    """
    Visualize feature sensitivity analysis (no shading).

    Parameters
    ----------
    results_df : pd.DataFrame
        Output from feature_sensitivity_analysis().
        Must contain columns: ['Model', 'NumFeatures', 'Score'].
    figsize : tuple, default=(12, 5)
        Figure size.
    save_path : str, optional
        Path to save the figure.
    metric_label : str, default="F1-score (tuned)"
        Y-axis label for the main plot.
    """

    # Make sure NumFeatures is numeric and sorted
    results_df = results_df.copy()
    results_df["NumFeatures"] = results_df["NumFeatures"].astype(int)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    # ----------------------------------------------------------------------
    # Plot 1: F1 vs number of features (per model)
    # ----------------------------------------------------------------------
    for model in results_df["Model"].unique():
        model_data = (
            results_df[results_df["Model"] == model]
            .sort_values("NumFeatures")
        )

        ax1.plot(
            model_data["NumFeatures"],
            model_data["Score"],
            marker="o",
            linewidth=2,
            label=model,
        )

    ax1.set_xlabel("Number of Features", fontsize=12)
    ax1.set_ylabel(metric_label, fontsize=12)
    ax1.set_title("Model Performance vs Feature Count", fontsize=14, fontweight="bold")
    ax1.grid(True, alpha=0.3)
    ax1.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),  # below the plot
    ncol=2,                       # spread across 2 columns
    fontsize=10
)

    # ----------------------------------------------------------------------
    # Plot 2: Best F1 per model
    # ----------------------------------------------------------------------
    # one row per model: the one with max Score
    best_df = (
        results_df.loc[results_df.groupby("Model")["Score"].idxmax()]
        .reset_index(drop=True)
    )

    bars = ax2.barh(best_df["Model"], best_df["Score"])

    for i, (score, n_feat) in enumerate(zip(best_df["Score"], best_df["NumFeatures"])):
        ax2.text(
            score + 0.01,
            i,
            f"{score:.3f}\n({n_feat} feat)",
            va="center",
            fontsize=10,
        )

    ax2.set_xlabel("Best F1-score", fontsize=12)
    ax2.set_title("Peak Performance Comparison", fontsize=14, fontweight="bold")
    ax2.grid(True, alpha=0.3, axis="x")

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

def plot_sensitivity_improvement(results_df):
    """
    Plot % improvement of F1 over the 1-feature baseline for each model.
    Assumes results_df has columns: 'Model', 'n_features', and 'F1-Score' (or 'F1').
    """
    models = results_df["Model"].unique()

    # Handle both possible F1 column names
    if "F1-Score" in results_df.columns:
        f1_col = "F1-Score"
    elif "Score" in results_df.columns:
        f1_col = "Score"
    else:
        raise ValueError("results_df must contain 'F1-Score' or 'F1' column.")

    plt.figure(figsize=(8, 5))

    for model in models:
        df_m = (
            results_df[results_df["Model"] == model]
            .sort_values("NumFeatures")
            .copy()
        )

        # Baseline = F1 with 1 feature
        base_row = df_m[df_m["NumFeatures"] == 1]
        if base_row.empty:
            # If you never evaluated NumFeatures == 1, skip this model
            continue

        base_f1 = base_row[f1_col].iloc[0]
        if base_f1 == 0:
            # Avoid division by zero; skip or handle differently if needed
            continue

        # Percentage improvement vs 1-feature baseline
        df_m["F1_improvement_pct"] = 100.0 * (df_m[f1_col] - base_f1) / base_f1

        plt.plot(
            df_m["NumFeatures"],
            df_m["F1_improvement_pct"],
            marker="o",
            label=model,
        )

    plt.axhline(0, color="gray", linestyle="--", linewidth=1)
    plt.xlabel("Number of Features")
    plt.ylabel("F1 Improvement over 1 Feature [%]")
    plt.title("Feature Sensitivity – F1 Improvement vs 1-Feature Baseline")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()



In [ ]:
clean_models = {
    name: d["model"]        # extract just the estimator
    for name, d in trained_models.items()
}
# X, y already defined; features already filtered (e.g. WV between 2–3 bar) in your current pipeline
ordered_features = [
    "Std_delay_exp", 
    "Total_power_delay",
    "Std_pipe",
    "Total_power_efficiency",
    "EmergencyBrake_action" 
]
selected_models = {
    "KNN (Imbalanced)": clean_models["KNN (Imbalanced)"],
    "Logistic Regression (Weighted)": clean_models["Logistic Regression (Weighted)"],
    "Decision Tree (Weighted)": clean_models["Decision Tree (Weighted)"],
    "XGBoost (Weighted)": clean_models["XGBoost (Weighted)"],
    "RandomForest (Weighted)": clean_models["Random Forest (Weighted)"]
}

[X_train, X_test, y_train, y_test, X_train_healthy] = preprocess_data(df, ordered_features, test_size=0.2)

[X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer] = scale_features(X_train, X_test, X_train_healthy)

# Run sensitivity analysis
results_df = feature_sensitivity_analysis(
    X=X_train_scaled,
    y=y_train,
    feature_names=ordered_features,
    models=selected_models,
    max_features=20,
    step=1  # Test every feature count
)

plot_sensitivity_analysis(results_df)


In [ ]:
plot_sensitivity_improvement(results_df)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_recall_curve, f1_score


def compare_feature_subsets_knn(
    X,
    y,
    feature_names,
    cv: int = 5,
    knn_kwargs: dict | None = None
):
    """
    Compare KNN performance with different feature subsets.

    For each subset of the first k features in `feature_names`, this function:
      - computes standard CV F1-score (using default KNN decision rule)
      - computes OOF probabilities and tunes the threshold to maximize F1
        (same logic as in cv_metrics_table)

    Assumes X is already scaled (e.g. by StandardScaler before calling).
    """
    if knn_kwargs is None:
        knn_kwargs = {"n_neighbors": 5, "weights": "uniform"}

    print("\n" + "="*70)
    print("KNN PERFORMANCE WITH DIFFERENT FEATURE SUBSETS (SCALED INPUT)")
    print("="*70)

    cv_splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    results = []

    for k in range(1, len(feature_names) + 1):
        subset = feature_names[:k]

        # Select subset of features
        if isinstance(X, pd.DataFrame):
            X_subset = X[subset].values
        else:
            indices = [feature_names.index(f) for f in subset]
            X_subset = X[:, indices]

        # -------------------------------------------------------------
        # 1) Standard CV F1 using default KNN decision rule
        # -------------------------------------------------------------
        knn = KNeighborsClassifier(**knn_kwargs)

        scores_default = cross_val_score(
            knn,
            X_subset,
            y,
            cv=cv_splitter,
            scoring="f1"
        )

        mean_f1_default = scores_default.mean()
        std_f1_default = scores_default.std()

        # -------------------------------------------------------------
        # 2) OOF probabilities + threshold tuning (tuned F1)
        # -------------------------------------------------------------
        # get out-of-fold predicted probabilities
        y_proba_oof = cross_val_predict(
            knn,
            X_subset,
            y,
            cv=cv_splitter,
            method="predict_proba"
        )[:, 1]

        precision_arr, recall_arr, thresholds = precision_recall_curve(y, y_proba_oof)
        f1_arr = 2 * precision_arr * recall_arr / (precision_arr + recall_arr + 1e-8)

        # thresholds has len = len(precision_arr) - 1 -> ignore last F1 point
        best_idx = np.argmax(f1_arr[:-1])
        best_thr = thresholds[best_idx]

        y_pred_tuned = (y_proba_oof >= best_thr).astype(int)
        f1_tuned = f1_score(y, y_pred_tuned, zero_division=0)

        results.append({
            "n_features": k,
            "features": subset[-1] if k > 0 else "",
            "mean_f1_default": mean_f1_default,
            "std_f1_default": std_f1_default,
            "f1_tuned": f1_tuned,
            "best_threshold": best_thr,
            "improvement": 0.0,   # filled below
        })

        print(
            f"{k} features | default F1: {mean_f1_default:.4f} ± {std_f1_default:.4f} "
            f"| tuned F1: {f1_tuned:.4f} (thr={best_thr:.3f}) "
            f"| Added: {subset[-1]}"
        )

    # Compute incremental improvement using tuned F1
    for i in range(1, len(results)):
        results[i]["improvement"] = results[i]["f1_tuned"] - results[i-1]["f1_tuned"]

    results_df = pd.DataFrame(results)

    print("\n" + "-"*70)
    print("FEATURE IMPACT (based on tuned F1):")
    print("-"*70)
    for i, row in results_df.iterrows():
        if i == 0:
            continue
        if row["improvement"] > 0.01:
            impact = "📈 POSITIVE"
        elif row["improvement"] < -0.01:
            impact = "📉 NEGATIVE"
        else:
            impact = "➡️  NEUTRAL"

        print(
            f"{impact} | Feature {row['n_features']}: {row['features']:30s} "
            f"| ΔF1_tuned = {row['improvement']:+.4f}"
        )

    return results_df


In [ ]:
knn_results = compare_feature_subsets_knn(
    X=X_train_scaled,
    y=y_train,
    feature_names=ordered_features
)

## Algorithm 2 - Multiple Features

### Feature Selection

In [ ]:
# ============================================================================
# STEP 1: DATA PREPROCESSING AND FEATURE SELECTION
# ============================================================================
selected_features = ['Total_power_efficiency','Std_delay_exp','Total_energy_efficiency']

[X_train, X_test, y_train, y_test, X_train_healthy] = preprocess_data(df, selected_features, test_size=0.2)
X_train.head()

In [ ]:
# ============================================================================
# STEP 2: IMPUTE + FEATURE SCALING
# ============================================================================

from sklearn.impute import SimpleImputer

def scale_features(X_train, X_test, X_train_healthy):
    """
    Impute missing values by median, then standardize features.
    Returns imputed+scaled arrays, plus fitted scaler and imputer.
    """
    # 1) Median imputation (fit only on training set)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)
    X_train_healthy_imp = imputer.transform(X_train_healthy)

    # 2) Standardization (fit only on imputed training set)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    X_train_healthy_scaled = scaler.transform(X_train_healthy_imp)

    return X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer

[X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer] = scale_features(X_train, X_test, X_train_healthy)

### Model Training and Cross Validation

In [ ]:
trained_models_multi_feat = train_all_models(X_train_scaled, y_train)


### Figure plotting of Cross Validation

In [ ]:
cv_summary_multi = cv_metrics_table(
    trained_models_multi_feat, X_train_scaled, y_train
)

cv_summary_multi

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# List of supervised models you want to visualize
best_models = [
    "KNN (Imbalanced)",
    "Logistic Regression (Weighted)",
    "Decision Tree (Weighted)",
    "Random Forest (Weighted)",
    "XGBoost (Weighted)"
]

# Convert test set to numpy
X = np.asarray(X_test_scaled)
y = np.asarray(y_test)

# --------------------------------------------------------
# LOOP THROUGH EACH MODEL AND CREATE A SEPARATE FIGURE
# --------------------------------------------------------
for model_name in best_models:

    # Retrieve model info
    model_info = trained_models_multi_feat[model_name]
    mtype = model_info['type']
    
    if mtype not in ('supervised', 'supervised_smote'):
        raise ValueError(f"{model_name} is not a supervised model.")

    base_model = model_info['model']

    # Retrieve best threshold from your CV summary table
    row = cv_summary_multi.loc[
        cv_summary_multi['Model'] == model_name
    ]
    if row.empty:
        raise ValueError(f"No CV summary row found for '{model_name}'.")

    best_threshold = row['BestThreshold'].iloc[0]

    # Predict on test set
    y_proba = base_model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= best_threshold).astype(int)

    # Confusion matrix
    cm = confusion_matrix(y, y_pred)
    max_count = cm.max()  # ensures consistent color scale

    # --------------------------------------------------------
    # PLOT — ONE FIGURE PER MODEL
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=(5, 4))

    im = ax.imshow(cm, cmap='coolwarm', vmin=0, vmax=max_count)

    ax.set_title(
        f"{model_name}\nThreshold = {best_threshold:.3f}",
        fontsize=12, pad=10
    )

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(['Pred Healthy', 'Pred Leakage'])
    ax.set_yticklabels(['True Healthy', 'True Leakage'])
    ax.grid(False)
    # Annotate values
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > max_count / 2 else 'black'
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color=color, fontsize=11)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Count")

    plt.tight_layout()
    plt.show()


### Precision Recall curves (Sensitive to imbalance data)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.base import clone

# -------------------------------------------------------
# 1. Choose models to plot
# -------------------------------------------------------
models_to_plot = [
    "Random Forest (Weighted)",
    "Decision Tree (Weighted)",
    "Logistic Regression (Weighted)",
    "KNN (Imbalanced)",
]

X = np.asarray(X_train_scaled)
y = np.asarray(y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

plt.figure(figsize=(7, 6))

for name in models_to_plot:
    entry = trained_models_multi_feat[name]
    base_model = entry['model']
    mtype      = entry['type']

    # --- build estimator depending on type ---
    if mtype == 'supervised':
        estimator = clone(base_model)

    elif mtype == 'supervised_smote':
        estimator = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', clone(base_model))
        ])

    else:
        print(f"Skipping {name} (type '{mtype}' not supported for PR curve).")
        continue

    # --- cross-validated scores (out-of-fold) ---
    y_scores = cross_val_predict(
        estimator, X, y,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    precision, recall, _ = precision_recall_curve(y, y_scores)
    ap = average_precision_score(y, y_scores)

    plt.plot(recall, precision, label=f"{name} (AP = {ap:.3f})")

# baseline: proportion of positives (random classifier)
pos_ratio = y.mean()
plt.hlines(pos_ratio, 0, 1, colors='gray', linestyles='--',
           label=f"Baseline (pos ratio = {pos_ratio:.2f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Cross-validated Precision–Recall Curves")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


### ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import cross_val_predict

plt.figure(figsize=(7, 6))

models_to_plot = [
    "Random Forest (Weighted)",
    "Decision Tree (Weighted)",
    "Logistic Regression (Weighted)",
    "KNN (Imbalanced)",
]

X = np.asarray(X_train_scaled)
y = np.asarray(y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name in models_to_plot:
    entry = trained_models_multi_feat[name]
    base_model = entry['model']
    mtype      = entry['type']

    # Build estimator
    if mtype == 'supervised':
        estimator = clone(base_model)
    elif mtype == 'supervised_smote':
        estimator = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', clone(base_model))
        ])
    else:
        print(f"Skipping anomaly model: {name}")
        continue

    # CV predicted probabilities
    y_scores = cross_val_predict(
        estimator, X, y,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    fpr, tpr, _ = roc_curve(y, y_scores)
    auc = roc_auc_score(y, y_scores)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", lw=2)

# Random baseline
plt.plot([0, 1], [0, 1], 'k--', label="Random classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Cross-validated ROC Curves for Multiple Models")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


### Comparison Table of F1 Score, Precision, Recall, and ROC value

### Tuning of 2 Features

In [ ]:
best_lr, lr_params, lr_f1, lr_cv_multi = tune_logistic_regression_weighted(X_train_scaled, y_train)
best_knn, knn_params, knn_f1, knn_cv_multi = tune_knn_classifier(X_train_scaled, y_train)
best_dt, dt_params, dt_f1, dt_cv_multi = tune_decision_tree_weighted(X_train_scaled, y_train)


In [ ]:
# Run XGboost tuning
best_xgb_est, best_xgb_params, best_xgb_score, xgb_cv_multi = tune_xgboost_weighted(
    X_train_scaled, y_train
)
# Run Random Forest tuning
best_rf_est, best_rf_params, best_rf_score, rf_cv_multi = tune_random_forest_weighted(
    X_train_scaled, y_train
)

In [ ]:
# Replace only the model with the best estimator
trained_models_multi_tuned = trained_models_multi_feat.copy()
trained_models_multi_tuned['XGBoost (Weighted)']['model'] = best_xgb_est
trained_models_multi_tuned['Random Forest (Weighted)']['model'] = best_rf_est
trained_models_multi_tuned['Logistic Regression (Weighted)']['model'] = best_lr
trained_models_multi_tuned['KNN (Imbalanced)']['model'] = best_knn
trained_models_multi_tuned['Decision Tree (Weighted)']['model'] = best_dt

# Evaluate
cv_results_multi_tuned = cv_metrics_table(
    trained_models_multi_tuned, X_train_scaled, y_train
)
cv_results_multi_tuned

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# Get the trained decision tree model
tree = trained_models_multi_tuned['Decision Tree (Weighted)']['model']

# Plot the tree
plt.figure(figsize=(16, 10))  # Adjust size as needed
plot_tree(
    tree,
    feature_names=selected_features,
    class_names=['Healthy', 'Leakage'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree Visualization")
plt.show()

In [ ]:
def fit_models(models_dict, X_train, y_train):
    for name, entry in models_dict.items():
        print(f"Training {name}...")
        entry['model'].fit(X_train, y_train)
    return models_dict

# Usage
trained_models_multi_tuned = fit_models(trained_models_multi_tuned, X_train_scaled, y_train)

In [ ]:
# On test set (no CV, reuse best thresholds)
test_results_multi = test_metrics_table(trained_models_multi_tuned, cv_results_multi_tuned, X_test_scaled, y_test)
test_results_multi

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# List of supervised models you want to visualize
best_models = [
    "KNN (Imbalanced)",
    "Logistic Regression (Weighted)",
    "Decision Tree (Weighted)",
    "Random Forest (Weighted)",
    "XGBoost (Weighted)"
]

# Convert test set to numpy
X = np.asarray(X_test_scaled)
y = np.asarray(y_test)

# --------------------------------------------------------
# LOOP THROUGH EACH MODEL AND CREATE A SEPARATE FIGURE
# --------------------------------------------------------
for model_name in best_models:

    # Retrieve model info
    model_info = trained_models_multi_tuned[model_name]
    mtype = model_info['type']
    
    if mtype not in ('supervised', 'supervised_smote'):
        raise ValueError(f"{model_name} is not a supervised model.")

    base_model = model_info['model']

    # Retrieve best threshold from your CV summary table
    row = cv_results_multi_tuned.loc[
        cv_results_multi_tuned['Model'] == model_name
    ]
    if row.empty:
        raise ValueError(f"No CV summary row found for '{model_name}'.")

    best_threshold = row['BestThreshold'].iloc[0]

    # Predict on test set
    y_proba = base_model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= best_threshold).astype(int)

    # Confusion matrix
    cm = confusion_matrix(y, y_pred)
    max_count = cm.max()  # ensures consistent color scale

    # --------------------------------------------------------
    # PLOT — ONE FIGURE PER MODEL
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=(5, 4))

    im = ax.imshow(cm, cmap='coolwarm', vmin=0, vmax=max_count)

    ax.set_title(
        f"{model_name}\nThreshold = {best_threshold:.3f}",
        fontsize=12, pad=10
    )

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(['Pred Healthy', 'Pred Leakage'])
    ax.set_yticklabels(['True Healthy', 'True Leakage'])
    ax.grid(False)
    # Annotate values
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > max_count / 2 else 'black'
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color=color, fontsize=11)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Count")

    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Clean model names if needed (important for merging)
df_before = cv_summary_multi.copy()
df_after = cv_results_multi_tuned.copy()

df_before['Model'] = df_before['Model'].str.strip()
df_after['Model'] = df_after['Model'].str.strip()

# Merge by model name
df_compare = pd.merge(
    df_before,
    df_after,
    on='Model',
    suffixes=('_before', '_after')
)

metrics = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC']

for m in metrics:
    df_compare[f'{m}_delta'] = df_compare[f'{m}_after'] - df_compare[f'{m}_before']

cols_show = [
    'Model',
    'BestThreshold_before', 'BestThreshold_after',
    'Precision_before', 'Precision_after', 'Precision_delta',
    'Recall_before', 'Recall_after', 'Recall_delta',
    'F1-Score_before', 'F1-Score_after', 'F1-Score_delta',
    'ROC-AUC_before', 'ROC-AUC_after', 'ROC-AUC_delta',
]

comparison_table = df_compare[cols_show].sort_values(by='F1-Score_delta', ascending=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

metrics = ['Precision', 'Recall', 'F1-Score']

# x locations for the groups
models = df_compare['Model'].tolist()
x = np.arange(len(models))
width = 0.35  # bar width

for metric in metrics:
    before_vals = df_compare[f'{metric}_before']
    after_vals  = df_compare[f'{metric}_after']

    plt.figure(figsize=(10, 5))
    
    # Bars: before and after, side by side
    plt.bar(x - width/2, before_vals, width, label='Before tuning')
    plt.bar(x + width/2, after_vals,  width, label='After tuning')
    
    plt.xticks(x, models, rotation=45, ha='right')
    plt.ylabel(metric)
    plt.title(f'{metric} (Training CV) – Before vs After Tuning')
    plt.ylim(0, 1.05)  # assuming metrics in [0, 1]
    plt.legend()
    plt.tight_layout()
    plt.show()


# Trained Model Comparison

## Improvement using multiple features

In [ ]:
cv_results_1_feat_tuned['Features'] = '1'
cv_results_multi_tuned['Features'] = '2'

# Merge on Model
comparison = cv_results_1_feat_tuned.merge(
    cv_results_multi_tuned,
    on="Model",
    suffixes=("_1feat", "_multi_feat")
)

# Keep only relevant metrics
metrics = ["Precision", "Recall", "F1-Score", "ROC-AUC"]
for m in metrics:
    comparison[f"{m}_Improvement"] = (
        comparison[f"{m}_multi_feat"] - comparison[f"{m}_1feat"]
    )
    
for m in metrics:
    comparison[f"{m}_PctImprovement"] = (
        (comparison[f"{m}_multi_feat"] - comparison[f"{m}_1feat"]) /
        comparison[f"{m}_1feat"] * 100
    )
import matplotlib.pyplot as plt

# Example: F1-score comparison
comparison.plot(
    x="Model",
    y=["F1-Score_1feat", "F1-Score_multi_feat"],
    kind="bar",
    figsize=(10,6)
)
plt.title("F1-Score Comparison (1 vs multi Features)")
plt.ylabel("F1-Score")
plt.xticks(rotation=45, ha="right")
plt.show()

# Example: Improvement only
comparison.plot(
    x="Model",
    y="F1-Score_Improvement",
    kind="bar",
    color="skyblue",
    figsize=(10,6)
)
plt.title("Improvement in F1-Score (multi Features vs 1 Feature)")
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("Δ F1-Score")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
metrics = ["Precision", "Recall", "F1-Score", "ROC-AUC"]

# Loop through each metric and plot percentage improvement
for m in metrics:
    plt.figure(figsize=(10,6))
    plt.bar(comparison["Model"], comparison[f"{m}_PctImprovement"], color="skyblue")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.title(f"Percentage Improvement in {m} (Multi Features vs 1 Feature)")
    plt.ylabel("% Improvement")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Clean model names if needed (important for merging)
df_training = cv_results_multi_tuned.copy()
df_predict = test_results_multi.copy()

df_training['Model'] = df_training['Model'].str.strip()
df_predict['Model'] = df_predict['Model'].str.strip()

# Merge by model name
df_compare = pd.merge(
    df_training,
    df_predict,
    on='Model',
    suffixes=('_training', '_predict')
)

metrics = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC']

for m in metrics:
    df_compare[f'{m}_delta'] = df_compare[f'{m}_predict'] - df_compare[f'{m}_training']

cols_show = [
    'Model',
    'BestThreshold_training', 'BestThreshold_predict',
    'Precision_training', 'Precision_predict', 'Precision_delta',
    'Recall_training', 'Recall_predict', 'Recall_delta',
    'F1-Score_training', 'F1-Score_predict', 'F1-Score_delta',
    'ROC-AUC_training', 'ROC-AUC_predict', 'ROC-AUC_delta',
]

comparison_table = df_compare[cols_show].sort_values(by='F1-Score_delta', ascending=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

metrics = ['Precision', 'Recall', 'F1-Score']

# x locations for the groups
models = df_compare['Model'].tolist()
x = np.arange(len(models))
width = 0.35  # bar width

for metric in metrics:
    before_vals = df_compare[f'{metric}_training']
    after_vals  = df_compare[f'{metric}_predict']

    plt.figure(figsize=(10, 5))
    
    # Bars: before and after, side by side
    plt.bar(x - width/2, before_vals, width, 
            label='Training performance', color='orange')
    plt.bar(x + width/2, after_vals,  width, 
            label='Test performance', color='purple')
    
    plt.xticks(x, models, rotation=45, ha='right')
    plt.ylabel(metric)
    plt.title(f'{metric} Training vs Test Performance')
    plt.ylim(0, 1.05)  # assuming metrics in [0, 1]
    plt.legend()
    plt.tight_layout()
    plt.show()